# IMPORT THƯ VIỆN VÀ DEF FUNCTION

In [1]:
import gc
import re
from pathlib import Path
import polars as pl

# -----------------------------------------------------------------------------
# HÀM XỬ LÝ CHUỖI NGÀY THÁNG (ĐÃ TỐI ƯU & BẢO VỆ)
# -----------------------------------------------------------------------------
def parse_multi_date(col_name: str) -> pl.Expr:
    col_expr = pl.col(col_name)
    
    # 1. Chuẩn hóa chuỗi (chỉ áp dụng nếu là chuỗi)
    clean_str = (
        col_expr
        .cast(pl.String)
        .str.strip_chars()
        .str.replace_all("/", "-")
    )

    # 2. Parse đa định dạng
    parsed_datetime = pl.coalesce([
        # Nếu cột vốn đã là Datetime/Date thì giữ nguyên
        col_expr.cast(pl.Datetime, strict=False),
        
        # Parse chuỗi dạng Năm - Ngày - Tháng
        clean_str.str.to_datetime("%Y-%d-%m %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%d-%m", strict=False),

        # Parse chuỗi dạng Năm - Tháng - Ngày
        clean_str.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d %H:%M", strict=False),
        clean_str.str.to_datetime("%Y-%m-%d", strict=False),

        # Parse chuỗi dạng Ngày - Tháng - Năm (Việt Nam)
        clean_str.str.to_datetime("%d-%m-%Y %H:%M:%S", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y %H:%M", strict=False),
        clean_str.str.to_datetime("%d-%m-%Y", strict=False),
    ])

    return parsed_datetime

def ultimate_to_date(col_name, df_context):
    if col_name not in df_context.columns:
        return pl.lit(None, dtype=pl.Date)

    col_str = pl.col(col_name).cast(pl.String)

    # Đọc chuỗi (dù là ISO, YYYY-MM-DD HH:MM:SS hay DD/MM/YYYY) và ép thẳng về Date
    parsed_date = pl.coalesce([
        col_str.str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
        col_str.str.slice(0, 10).str.to_date("%d/%m/%Y", strict=False),
        col_str.str.slice(0, 10).str.to_date("%d-%m-%Y", strict=False),
    ])

    excel_date = (
        pl.col(col_name)
        .cast(pl.Float64, strict=False)
        .cast(pl.Duration("ms"))
        + pl.date(1899, 12, 30)
    ).dt.date()

    return pl.coalesce([parsed_date, excel_date])

def ultimate_to_datetime(col_name, df_context):
    if col_name not in df_context.columns:
        return pl.lit(None, dtype=pl.Datetime)

    col_str = pl.col(col_name).cast(pl.String)

    # 1. Parse các định dạng chuỗi phổ biến sang Datetime (phải có giờ:phút:giây)
    # Lưu ý: slice(0, 19) để lấy đủ chuỗi dạng "YYYY-MM-DD HH:MM:SS"
    col_str_19 = col_str.str.slice(0, 19)
    
    parsed_dt = pl.coalesce([
        # Định dạng chuẩn ISO / SQL
        col_str_19.str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
        col_str.str.to_datetime("%Y-%m-%d", strict=False),
        
        # Định dạng DD/MM/YYYY
        col_str_19.str.to_datetime("%d/%m/%Y %H:%M:%S", strict=False),
        col_str.str.to_datetime("%d/%m/%Y", strict=False),
        
        # Định dạng DD-MM-YYYY
        col_str_19.str.to_datetime("%d-%m-%Y %H:%M:%S", strict=False),
        col_str.str.to_datetime("%d-%m-%Y", strict=False),
    ])

    # 2. Xử lý Excel Serial Date (1 ngày = 86,400,000 miligiây) -> giữ cả phần lẻ giờ:phút:giây
    excel_dt = (
        (pl.col(col_name).cast(pl.Float64, strict=False) * 86_400_000)
        .cast(pl.Duration("ms"))
        + pl.datetime(1899, 12, 30)
    )

    return pl.coalesce([parsed_dt, excel_dt])

# TTS ĐƠN PHÁT

In [2]:
# -----------------------------------------------------------------------------
# 1. CẤU HÌNH ĐƯỜNG DẪN & CỘT CẦN LẤY
# -----------------------------------------------------------------------------
path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat")

selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "tg_quydinh", "ngay_gui_bp",
    "time_pcp", "tg_quydinhphat", "tg_chenhlechphat", "ma_dv_viettel", "ma_buucuc_goc",
    "ma_buucuc_phat", "ma_trangthai", "ma_doitac", "ma_khgui", "tg_toantrinh",
    "danhgia_time_gach_bp1", "danhgia_time_gach_bp2", "danhgia_time_gach_bp3",
    "time_gach_bp", "time_gach_bp2", "time_gach_bp3", "tg_ptc", "tg_nhantai_bcp", "trong_luong","KHAU_SAI",
    "tien_cod",	"tienhang",	"tong_cuoc"

]

date_cols = [
    "ngay_gui_bp", "tg_ptc", "tg_nhantai_bcp", "time_pcp", 
    "tg_quydinhphat", "time_gach_bp", "time_gach_bp2", "time_gach_bp3"
]

# -----------------------------------------------------------------------------
# 2. NẠP VÀ TỐI ƯU TRỰC TIẾP TỪNG FILE EXCEL
# -----------------------------------------------------------------------------
print("🚀 1/4. Đang nạp và xử lý từng file Excel...")

dfs = []
excel_files = list(path.glob("chitietketquaphat_TikTok_*.xlsx"))

if not excel_files:
    print("❌ Không tìm thấy file Excel nào khớp với mẫu!")
else:
    for file in excel_files:
        # Đọc file với mẫu 10,000 dòng để đạt tốc độ cao nhất
        df = pl.read_excel(file, drop_empty_rows=False, infer_schema_length=10000)
        
        # Chỉ giữ lại các cột cần thiết ngay từ đầu để tiết kiệm RAM
        existing_cols = [c for c in selected_columns if c in df.columns]
        df = df.select(existing_cols)
        
        # Parse ngày trực tiếp trên từng file nhỏ (Truyền c dạng chuỗi str)
        date_exprs = [parse_multi_date(c).alias(c) for c in date_cols if c in df.columns]
        if date_exprs:
            df = df.with_columns(date_exprs)
            
        df = df.with_columns(pl.lit(file.name).alias("Ten_File_Nguon"))
        dfs.append(df)

    print("⚡ 2/4. Đang gộp các bảng dữ liệu...")
    df_final = pl.concat(dfs, how="diagonal_relaxed")
    
    del dfs
    gc.collect()

# -----------------------------------------------------------------------------
# 3. TRÍCH XUẤT VÀ CHUẨN HÓA DỮ LIỆU
# -----------------------------------------------------------------------------
print("🛠️ 3/4. Đang trích xuất thông tin...")

# Ghép tuyến
df_final = df_final.with_columns(
    pl.concat_str([pl.col("tinh_nhan"), pl.lit("->"), pl.col("tinh_phat")]).alias("tuyen")
)

# Trích xuất số từ tg_chenhlechphat
df_final = df_final.with_columns(
    pl.col("tg_chenhlechphat")
    .cast(pl.String)
    .str.extract(r"(-?\d+)", 1)
    .cast(pl.Int64, strict=False)
    .alias("tg_chenhlechphat_so")
)

# -----------------------------------------------------------------------------
# 4. TÍNH LOGIC VÀ PHÂN LOẠI
# -----------------------------------------------------------------------------
print("🧠 4/4. Đang tính toán cờ KPI và phân loại giao hàng...")

df_final = (
    df_final.with_columns([
        # Ngày bắt đầu phải phát
        pl.when(pl.col("tg_nhantai_bcp").is_not_null())
        .then(pl.col("tg_nhantai_bcp"))
        .when(pl.col("time_pcp").is_not_null())
        .then(pl.col("time_pcp"))
        .otherwise(pl.col("ngay_gui_bp"))
        .alias("ngay_bat_dau_phai_phat"),

        # Ngày phát cuối cùng
        pl.when(pl.col("tg_ptc").is_not_null())
        .then(pl.col("tg_ptc"))
        .otherwise(
            pl.max_horizontal([
                pl.col("time_gach_bp"),
                pl.col("time_gach_bp2"),
                pl.col("time_gach_bp3"),
            ])
        )
        .alias("ngay_phat_cuoi_cung"),

        # Cờ PTC
        pl.when(pl.col("tg_ptc").is_not_null()).then(1).otherwise(0).alias("PTC"),

        # Cờ PTC_1
        pl.when(
            (pl.col("tg_ptc") == pl.col("time_gach_bp"))
            & (pl.col("danhgia_time_gach_bp1") == "Đúng chỉ tiêu")
        )
        .then(1)
        .otherwise(0)
        .alias("PTC_1"),

        # Đánh giá giao hàng
        pl.when(pl.col("tg_chenhlechphat_so").is_null())
        .then(pl.lit("Không xác định"))
        .when(pl.col("tg_chenhlechphat_so") > 0)
        .then(pl.lit("Giao không đúng giờ"))
        .otherwise(pl.lit("Giao đúng giờ"))
        .alias("danh_gia_giao_hang"),
    ])
    .filter(pl.col("ngay_bat_dau_phai_phat").is_not_null())
)

print("=" * 70)
print(f"✅ HOÀN THÀNH! Tổng số bản ghi đã xử lý: {df_final.height:,}")
print("=" * 70)

🚀 1/4. Đang nạp và xử lý từng file Excel...


Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 37, falling back to string
Could not determine dtype for column 38, falling back to string


⚡ 2/4. Đang gộp các bảng dữ liệu...
🛠️ 3/4. Đang trích xuất thông tin...
🧠 4/4. Đang tính toán cờ KPI và phân loại giao hàng...
✅ HOÀN THÀNH! Tổng số bản ghi đã xử lý: 448,235


In [3]:
# -----------------------------------------------------------------------------
# TỔNG HỢP VÀ EXPLODE KHOẢNG NGÀY
# -----------------------------------------------------------------------------
selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat",
    "ma_trangthai", "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang",
    "trong_luong", "Ten_File_Nguon","tg_ptc","KHAU_SAI","tien_cod",	"tienhang",	"tong_cuoc","danhgia_time_gach_bp1"
]

existing_cols = [col for col in selected_columns if col in df_final.columns]
df_raw = df_final.select(existing_cols)
df_exploded = df_raw

In [4]:
df_phat_processed = df_exploded.with_columns([
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])

index_cols = [
    "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat", "ma_trangthai","ma_phieugui",
    "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang","KHAU_SAI",
    # "ngay_trong_khoang",
    "nhom_trong_luong", "Ten_File_Nguon","tg_ptc", "tien_cod",	"tienhang",	"trong_luong","tong_cuoc","danhgia_time_gach_bp1"
]

cols_exist = [c for c in index_cols if c in df_phat_processed.columns]
df_phat_processed = df_phat_processed.select(cols_exist)

In [5]:
df_phat_processed = df_phat_processed.filter(pl.col("ma_trangthai") == "501")
df_phat_processed = df_phat_processed.filter(pl.col("tg_ptc").is_not_null())

In [6]:
# 1. Khởi tạo cột KHAU_SAI với giá trị mặc định là "Chưa xác định"
df_final_test = df_phat_processed.with_columns(
    pl.lit("Chưa xác định").alias("KHAU_SAI")
)

# 2. Lấy danh sách cột hiện tại và di chuyển KHAU_SAI vào ngay sau tg_ptc
cols = df_final_test.columns

if "KHAU_SAI" in cols:
    cols.remove("KHAU_SAI")

idx = cols.index("tg_ptc") + 1
cols.insert(idx, "KHAU_SAI")

# 3. Áp dụng thứ tự cột mới cho DataFrame
df_final_test = df_final_test.select(cols)

In [7]:
path_tts_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat")
# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "TTS_st_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_tts_path) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_final_test.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")

Đã lưu file Parquet thành công tại: C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat\TTS_st_new_today.parquet


In [10]:
import os
import shutil

# 1. Khai báo đường dẫn thư mục và file
folder_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Phat")

file_phat = folder_path / "TTS_st_phat_database.parquet"   # File database chính
file_new = folder_path / "TTS_st_new_today.parquet"        # File mới cần gộp
file_temp = folder_path / "TTS_st_phat_database_TEMP.parquet" # File tạm tránh lock Windows
file_backup = folder_path / "TTS_st_phat_database_BACKUP.parquet" # File backup an toàn

# 2. Kiểm tra sự tồn tại của file new
if not file_new.exists():
    print(f"⚠️ Không tìm thấy file new: {file_new.name}")
else:
    # --- BƯỚC BẢO VỆ: Tạo bản backup tạm thời trước khi gộp ---
    if file_phat.exists():
        shutil.copy(file_phat, file_backup)

    list_dfs = []

    # Đọc file database cũ (nếu có)
    if file_phat.exists():
        df_phat = pl.read_parquet(file_phat)
        list_dfs.append(df_phat)

    # Đọc file new
    df_new = pl.read_parquet(file_new)
    list_dfs.append(df_new)

    # 3. Gộp dữ liệu
    df_gop = pl.concat(list_dfs, how="diagonal")

    # 4. Ghi ra file tạm (tránh lỗi lock os error 1224)
    df_gop.write_parquet(file_temp)

    # 5. Thay thế file tạm thành file chính
    os.replace(file_temp, file_phat)
    print(f"✅ Đã gộp thành công dữ liệu vào file: {file_phat.name}")

    # 6. DỌN DẸP: Xóa file st_new và xóa luôn file backup sau khi đã gộp thành công
    if file_new.exists():
        os.remove(file_new)
        print(f"🗑️ Đã xóa file nguồn: {file_new.name}")

    if file_backup.exists():
        os.remove(file_backup)
        print(f"🗑️ Đã dọn dẹp file backup an toàn!")

Đã tạo 1 bản backup dự phòng an toàn!
Đã gộp thành công dữ liệu vào file: TTS_st_phat_database.parquet
Đã xóa file nguồn: TTS_st_new_today.parquet


# SPE ĐƠN PHÁT

In [11]:
# -----------------------------------------------------------------------------
# 1. CẤU HÌNH ĐƯỜNG DẪN & CỘT CẦN LẤY
# -----------------------------------------------------------------------------
path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Phat")

selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "tg_quydinh", "ngay_gui_bp",
    "time_pcp", "tg_quydinhphat", "tg_chenhlechphat", "ma_dv_viettel", "ma_buucuc_goc",
    "ma_buucuc_phat", "ma_trangthai", "ma_doitac", "ma_khgui", "tg_toantrinh",
    "danhgia_time_gach_bp1", "danhgia_time_gach_bp2", "danhgia_time_gach_bp3",
    "time_gach_bp", "time_gach_bp2", "time_gach_bp3", "tg_ptc", "tg_nhantai_bcp", "trong_luong","KHAU_SAI",
    "tien_cod",	"tienhang",	"tong_cuoc"

]

date_cols = [
    "ngay_gui_bp", "tg_ptc", "tg_nhantai_bcp", "time_pcp", 
    "tg_quydinhphat", "time_gach_bp", "time_gach_bp2", "time_gach_bp3"
]

# -----------------------------------------------------------------------------
# 2. NẠP VÀ TỐI ƯU TRỰC TIẾP TỪNG FILE EXCEL
# -----------------------------------------------------------------------------
print("🚀 1/4. Đang nạp và xử lý từng file Excel...")

dfs = []
excel_files = list(path.glob("chitietketquaphat_TikTok_*.xlsx"))

if not excel_files:
    print("❌ Không tìm thấy file Excel nào khớp với mẫu!")
else:
    for file in excel_files:
        # Đọc file với mẫu 10,000 dòng để đạt tốc độ cao nhất
        df = pl.read_excel(file, drop_empty_rows=False, infer_schema_length=10000)
        
        # Chỉ giữ lại các cột cần thiết ngay từ đầu để tiết kiệm RAM
        existing_cols = [c for c in selected_columns if c in df.columns]
        df = df.select(existing_cols)
        
        # Parse ngày trực tiếp trên từng file nhỏ (Truyền c dạng chuỗi str)
        date_exprs = [parse_multi_date(c).alias(c) for c in date_cols if c in df.columns]
        if date_exprs:
            df = df.with_columns(date_exprs)
            
        df = df.with_columns(pl.lit(file.name).alias("Ten_File_Nguon"))
        dfs.append(df)

    print("⚡ 2/4. Đang gộp các bảng dữ liệu...")
    df_final = pl.concat(dfs, how="diagonal_relaxed")
    
    del dfs
    gc.collect()

# -----------------------------------------------------------------------------
# 3. TRÍCH XUẤT VÀ CHUẨN HÓA DỮ LIỆU
# -----------------------------------------------------------------------------
print("🛠️ 3/4. Đang trích xuất thông tin...")

# Ghép tuyến
df_final = df_final.with_columns(
    pl.concat_str([pl.col("tinh_nhan"), pl.lit("->"), pl.col("tinh_phat")]).alias("tuyen")
)

# Trích xuất số từ tg_chenhlechphat
df_final = df_final.with_columns(
    pl.col("tg_chenhlechphat")
    .cast(pl.String)
    .str.extract(r"(-?\d+)", 1)
    .cast(pl.Int64, strict=False)
    .alias("tg_chenhlechphat_so")
)

# -----------------------------------------------------------------------------
# 4. TÍNH LOGIC VÀ PHÂN LOẠI
# -----------------------------------------------------------------------------
print("🧠 4/4. Đang tính toán cờ KPI và phân loại giao hàng...")

df_final = (
    df_final.with_columns([
        # Ngày bắt đầu phải phát
        pl.when(pl.col("tg_nhantai_bcp").is_not_null())
        .then(pl.col("tg_nhantai_bcp"))
        .when(pl.col("time_pcp").is_not_null())
        .then(pl.col("time_pcp"))
        .otherwise(pl.col("ngay_gui_bp"))
        .alias("ngay_bat_dau_phai_phat"),

        # Ngày phát cuối cùng
        pl.when(pl.col("tg_ptc").is_not_null())
        .then(pl.col("tg_ptc"))
        .otherwise(
            pl.max_horizontal([
                pl.col("time_gach_bp"),
                pl.col("time_gach_bp2"),
                pl.col("time_gach_bp3"),
            ])
        )
        .alias("ngay_phat_cuoi_cung"),

        # Cờ PTC
        pl.when(pl.col("tg_ptc").is_not_null()).then(1).otherwise(0).alias("PTC"),

        # Cờ PTC_1
        pl.when(
            (pl.col("tg_ptc") == pl.col("time_gach_bp"))
            & (pl.col("danhgia_time_gach_bp1") == "Đúng chỉ tiêu")
        )
        .then(1)
        .otherwise(0)
        .alias("PTC_1"),

        # Đánh giá giao hàng
        pl.when(pl.col("tg_chenhlechphat_so").is_null())
        .then(pl.lit("Không xác định"))
        .when(pl.col("tg_chenhlechphat_so") > 0)
        .then(pl.lit("Giao không đúng giờ"))
        .otherwise(pl.lit("Giao đúng giờ"))
        .alias("danh_gia_giao_hang"),
    ])
    .filter(pl.col("ngay_bat_dau_phai_phat").is_not_null())
)

print("=" * 70)
print(f"✅ HOÀN THÀNH! Tổng số bản ghi đã xử lý: {df_final.height:,}")
print("=" * 70)

🚀 1/4. Đang nạp và xử lý từng file Excel...


Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 32, falling back to string
Could not determine dtype for column 35, falling back to string
Could not determine dtype for column 36, falling back to string
Could not determine dtype for column 37, falling back to string
Could not determine dtype for column 38, falling back to string


⚡ 2/4. Đang gộp các bảng dữ liệu...
🛠️ 3/4. Đang trích xuất thông tin...
🧠 4/4. Đang tính toán cờ KPI và phân loại giao hàng...
✅ HOÀN THÀNH! Tổng số bản ghi đã xử lý: 117,064


In [12]:
# -----------------------------------------------------------------------------
# TỔNG HỢP VÀ EXPLODE KHOẢNG NGÀY
# -----------------------------------------------------------------------------
selected_columns = [
    "ma_phieugui", "tinh_nhan", "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat",
    "ma_trangthai", "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang",
    "trong_luong", "Ten_File_Nguon","tg_ptc","KHAU_SAI","tien_cod",	"tienhang",	"tong_cuoc","danhgia_time_gach_bp1"
]

existing_cols = [col for col in selected_columns if col in df_final.columns]
df_raw = df_final.select(existing_cols)
df_exploded = df_raw

In [13]:
df_phat_processed = df_exploded.with_columns([
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])

index_cols = [
    "tinh_phat", "ma_dv_viettel", "ma_buucuc_phat", "ma_trangthai","ma_phieugui",
    "ma_doitac", "ma_khgui", "tuyen", "PTC", "PTC_1",
    "ngay_bat_dau_phai_phat", "ngay_phat_cuoi_cung", "danh_gia_giao_hang","KHAU_SAI",
    # "ngay_trong_khoang",
    "nhom_trong_luong", "Ten_File_Nguon","tg_ptc", "tien_cod",	"tienhang",	"trong_luong","tong_cuoc","danhgia_time_gach_bp1"
]

cols_exist = [c for c in index_cols if c in df_phat_processed.columns]
df_phat_processed = df_phat_processed.select(cols_exist)

In [14]:
df_phat_processed = df_phat_processed.filter(pl.col("ma_trangthai") == "501")
df_phat_processed = df_phat_processed.filter(pl.col("tg_ptc").is_not_null())

In [15]:
# 1. Khởi tạo cột KHAU_SAI với giá trị mặc định là "Chưa xác định"
df_final_test = df_phat_processed.with_columns(
    pl.lit("Chưa xác định").alias("KHAU_SAI")
)

# 2. Lấy danh sách cột hiện tại và di chuyển KHAU_SAI vào ngay sau tg_ptc
cols = df_final_test.columns

if "KHAU_SAI" in cols:
    cols.remove("KHAU_SAI")

idx = cols.index("tg_ptc") + 1
cols.insert(idx, "KHAU_SAI")

# 3. Áp dụng thứ tự cột mới cho DataFrame
df_final_test = df_final_test.select(cols)

In [16]:
path_spe_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Phat")
# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "SPE_st_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_spe_path) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_final_test.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")

Đã lưu file Parquet thành công tại: C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Phat\SPE_st_new_today.parquet


In [17]:
import os
import shutil

# 1. Khai báo đường dẫn thư mục và file
folder_path = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Phat")

file_phat = folder_path / "SPE_st_phat_database.parquet"   # File database chính
file_new = folder_path / "SPE_st_new_today.parquet"        # File mới cần gộp
file_temp = folder_path / "SPE_st_phat_database_TEMP.parquet" # File tạm tránh lock Windows
file_backup = folder_path / "SPE_st_phat_database_BACKUP.parquet" # File backup an toàn

# 2. Kiểm tra sự tồn tại của file new
if not file_new.exists():
    print(f"⚠️ Không tìm thấy file new: {file_new.name}")
else:
    # --- BƯỚC BẢO VỆ: Tạo bản backup tạm thời trước khi gộp ---
    if file_phat.exists():
        shutil.copy(file_phat, file_backup)

    list_dfs = []

    # Đọc file database cũ (nếu có)
    if file_phat.exists():
        df_phat = pl.read_parquet(file_phat)
        list_dfs.append(df_phat)

    # Đọc file new
    df_new = pl.read_parquet(file_new)
    list_dfs.append(df_new)

    # 3. Gộp dữ liệu
    df_gop = pl.concat(list_dfs, how="diagonal")

    # 4. Ghi ra file tạm (tránh lỗi lock os error 1224)
    df_gop.write_parquet(file_temp)

    # 5. Thay thế file tạm thành file chính
    os.replace(file_temp, file_phat)
    print(f"✅ Đã gộp thành công dữ liệu vào file: {file_phat.name}")

    # 6. DỌN DẸP: Xóa file st_new và xóa luôn file backup sau khi đã gộp thành công
    if file_new.exists():
        os.remove(file_new)
        print(f"🗑️ Đã xóa file nguồn: {file_new.name}")

    if file_backup.exists():
        os.remove(file_backup)
        print(f"🗑️ Đã dọn dẹp file backup an toàn!")

✅ Đã gộp thành công dữ liệu vào file: SPE_st_phat_database.parquet
🗑️ Đã xóa file nguồn: SPE_st_new_today.parquet
🗑️ Đã dọn dẹp file backup an toàn!


# SPE THU

In [ ]:
path_thu = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Thu")

print("🚀 Đang nạp các file chitietketquathu_TikTok...")

dfs_thu = []
excel_files_thu = list(path_thu.glob("chitietketquathu_TikTok_*.xlsx"))

if not excel_files_thu:
    print("❌ Không tìm thấy file Excel nào!")
else:
    for file in excel_files_thu:
        df = pl.read_excel(file, drop_empty_rows=True)
        
        # 1. Ép TẤT CẢ các cột về String để tránh xung đột Schema khi concat
        # Polars sẽ đọc chuẩn hơn khi ép chuỗi toàn bộ trước khi gộp
        df = df.with_columns([pl.col(c).cast(pl.String) for c in df.columns])
        
        # df = df.with_columns(pl.lit(file.name).alias("Ten_File_Nguon"))
        dfs_thu.append(df)

    print("⚡ Đang gộp dữ liệu Thu...")
    df_thu_final = pl.concat(dfs_thu, how="diagonal")
    del dfs_thu
    gc.collect()

    # 2. Định danh và ép kiểu Ngày/Thời gian trên Dataframe đã gộp duy nhất
    date_cols = [c for c in df_thu_final.columns if any(k in c.lower() for k in ["ngay", "time", "tg", "thoigian"])]
    df_thu_final = df_thu_final.with_columns([
        ultimate_to_date(c, df_thu_final).alias(c) for c in date_cols
    ])

    print("=" * 70)
    print(f"✅ GỘP THÀNH CÔNG DỮ LIỆU THU! Tổng số dòng: {df_thu_final.height:,}")
    print("=" * 70)

In [ ]:
df_thu_processed = df_thu_final.with_columns([
    # Ép kiểu trọng lượng về số và phân nhóm
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])
df_thu_processed

In [ ]:
# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "Tiktok_thu_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_thu) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_thu_processed.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")


In [ ]:
df_gop = pl.scan_parquet(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Shopee\Thu\*.parquet").collect()

# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file_2 = "SPE_thu_database.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh_2 = Path(path_thu) / ten_file_2
duong_dan_hoan_chinh_2.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_gop.write_parquet(duong_dan_hoan_chinh_2)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh_2}")

# TTS THU

In [ ]:
path_thu = Path(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Thu")

print("🚀 Đang nạp các file chitietketquathu_TikTok...")

dfs_thu = []
excel_files_thu = list(path_thu.glob("chitietketquathu_TikTok_*.xlsx"))

if not excel_files_thu:
    print("❌ Không tìm thấy file Excel nào!")
else:
    for file in excel_files_thu:
        df = pl.read_excel(file, drop_empty_rows=True)
        
        # 1. Ép TẤT CẢ các cột về String để tránh xung đột Schema khi concat
        # Polars sẽ đọc chuẩn hơn khi ép chuỗi toàn bộ trước khi gộp
        df = df.with_columns([pl.col(c).cast(pl.String) for c in df.columns])
        
        # df = df.with_columns(pl.lit(file.name).alias("Ten_File_Nguon"))
        dfs_thu.append(df)

    print("⚡ Đang gộp dữ liệu Thu...")
    df_thu_final = pl.concat(dfs_thu, how="diagonal")
    del dfs_thu
    gc.collect()

    # 2. Định danh và ép kiểu Ngày/Thời gian trên Dataframe đã gộp duy nhất
    date_cols = [c for c in df_thu_final.columns if any(k in c.lower() for k in ["ngay", "time", "tg", "thoigian"])]
    df_thu_final = df_thu_final.with_columns([
        ultimate_to_date(c, df_thu_final).alias(c) for c in date_cols
    ])

    print("=" * 70)
    print(f"✅ GỘP THÀNH CÔNG DỮ LIỆU THU! Tổng số dòng: {df_thu_final.height:,}")
    print("=" * 70)

In [ ]:
df_thu_processed = df_thu_final.with_columns([
    # Ép kiểu trọng lượng về số và phân nhóm
    pl.when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 5000)
    .then(pl.lit("< 5kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 10000)
    .then(pl.lit("5kg - < 10kg"))
    .when(pl.col("trong_luong").cast(pl.Float64, strict=False) < 20000)
    .then(pl.lit("10kg - < 20kg"))
    .otherwise(pl.lit(">= 20kg"))
    .alias("nhom_trong_luong")
])
df_thu_processed

In [ ]:
# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file = "Tiktok_thu_new_today.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh = Path(path_thu) / ten_file
duong_dan_hoan_chinh.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_thu_processed.write_parquet(duong_dan_hoan_chinh)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh}")


In [ ]:
df_gop_2 = pl.scan_parquet(r"C:\Users\lamnv5_vtp\Downloads\Streamlit\Tiktok\Thu\*.parquet").collect()

# --- 1. Nhập đường dẫn thư mục và tên file bạn muốn ---
ten_file_2 = "TTS_thu_database.parquet"     # Tên file tùy ý bạn đặt

# --- 2. Tạo đường dẫn hoàn chỉnh và tự động tạo thư mục nếu chưa có ---
duong_dan_hoan_chinh_2 = Path(path_thu) / ten_file_2
duong_dan_hoan_chinh_2.parent.mkdir(parents=True, exist_ok=True)

# --- 3. Ghi file Parquet ---
df_gop_2.write_parquet(duong_dan_hoan_chinh_2)

print(f"Đã lưu file Parquet thành công tại: {duong_dan_hoan_chinh_2}")